# 3×3 Rydberg quench 当前可用后端 benchmark

这个 notebook 用同一套 3×3 `1r` 方格晶格和同一条失谐扫描 protocol，对当前仓库保留的模拟路径做 benchmark。
每个算法占三个 block，第一个 block 说明算法，第二个 block 运行并计时，第三个 block 画该算法的逐格点 Rydberg occupation 热图。

当前可用的 TN 公开入口只有：
- `backend="mps"`：TeNPy CPU MPS reference；
- `backend="peps"`：YASTN finite PEPS，二维 lattice 主路径；
- `backend="gputn"`：cuTensorNet/CuPy GPU experimental path。

在 `1r` 能级结构下，每个格点有两能级：`|1⟩`（基态）和 `|r⟩`（Rydberg）。对本 notebook 的 3×3 方格（9 格点、最近邻 VdW），时间依赖 Hamiltonian 为：

$$
H(t) = \sum_{i=0}^{8} \left[ \frac{\Omega(t)}{2}\,\sigma^x_i - \Delta_i(t)\, n_i^r \right] + \sum_{\langle i,j\rangle} V_{ij}\, n_i^r n_j^r
$$

其中：
- $\sigma^x_i = |r\rangle\langle 1|_i + |1\rangle\langle r|_i$
- $n_i^r = |r\rangle\langle r|_i$（Rydberg 占据算符）
- $\Delta_i(t) = \Delta(t) + \Delta_{\mathrm{addr},i}(t)$；
- $V_{ij} = C_6 / r_{ij}^6$，`mode="nn"` 只保留最近邻，在$r = 6 \mu m, 10 \mu m$时，Rydberg interaction分别是 $2\pi 	imes 18 MHz,2\pi 	imes  0.874 MHz$
- 角频率单位均为 rad/s

Sweep time不能选择太久，按照300K 下 70S Rydberg decay rate $\Gamma = 1/151.55 (\mu s)^{-1}$.

 如果想要控制每个原子的loss rate 在 $0.99$以下，至少需要 $t = -\ln(0.99) 	imes 151.55 \mu s pprox 1.523 \mu s$.


In [ ]:
import importlib
import time

import numpy as np
import matplotlib.pyplot as plt

# Refresh SweepProtocol when rerunning this notebook in an already-started kernel.
rg_sweep = importlib.reload(rg_sweep)
rg.SweepProtocol = rg_sweep.SweepProtocol

# Experimental-style parameters
a_um = 10                         # lattice spacing, um
Omega = 2 * np.pi * 3.8e6          # rad/s
C6_70s = 2 * np.pi * 874e9         # rad/s * um^6, Rb 70S typical

delta_start = -2 * np.pi * 10.0e6   # rad/s
delta_end = 2 * np.pi * 10.0e6      # rad/s
t_sweep = 1.5e-6                   # s
omega_ramp_frac = 0.1
N_sites = 3 * 3


这里使用两条不同形状的平滑波形，而不是让 Rabi 和 detuning 共用同一个 bump。

Rabi 用 smooth flat-top：先用 smootherstep 平滑打开，保持平台，再平滑关闭。取
$$
h(u)=10u^3-15u^4+6u^5,
$$
并令平台边缘宽度为 $\tau_r=0.09T$。这样在两次共振穿越附近，$\Omega(t)/2$ 已经处在平台上，avoided crossing 的 gap 足够大，第二次扫回时更容易把多体态带回基态。当前 $T=1.5\,\mu s$ 时，$\tau_r=0.135\,\mu s$；smootherstep 的 $10\%-90\%$ 上升时间约为 $0.5067\tau_r\approx 0.068\,\mu s$，粗略等效带宽为
$$
f_{\mathrm{req}}\sim 0.35/t_{10-90}\approx 5.1\,\mathrm{MHz}.
$$
这个量级通常可以由 AOM + analog RF driver/AWG 做到；实验上应先用 fast photodiode 测量实际光功率波形，再用 LUT 或 predistortion 校正 RF driver 和 AOM diffraction efficiency 的非线性。

Detuning 用余弦往返：
$$
\Delta(t)=\Delta_{\rm mid}-\Delta_{\rm amp}\cos(2\pi t/T),
$$
其中 $\Delta_{\rm mid}=(\Delta_{\rm start}+\Delta_{\rm end})/2$，$\Delta_{\rm amp}=(\Delta_{\rm end}-\Delta_{\rm start})/2$。它保持 $-10\to +10\to -10\,\mathrm{MHz}$ 的取值范围，并且端点斜率为零。实验上 detuning 波形通常通过 chirp AOM RF frequency、EOM/PLL，或直接调激光频率实现；AWG 应输出连续相位 $\phi(t)=2\pi\int f_{\rm RF}(t)dt$，而不是只逐点跳频。

AOM 用作 Rabi 光强调制时，控制链路近似为
$$
V_{\mathrm{AWG}}(t)\rightarrow P_{\mathrm{RF}}(t)\rightarrow P_{\mathrm{opt}}(t)\rightarrow \Omega(t).
$$
如果想要 $\Omega(t)\propto b(t)$，通常光功率目标应接近 $P_{\mathrm{opt}}(t)\propto b(t)^2$。因此代码里的 `omega_half_t` 是目标 Rabi envelope；实际 AWG 电压需要根据测得的 AOM transfer function 反推。

In [ ]:
import ryd_gate as rg
import ryd_gate.protocols.sweep as rg_sweep
from ryd_gate import InteractionSpec
from ryd_gate.lattice import make_square_lattice
from ryd_gate.backends.tn_common.compiler import tn_lattice_spec_from_system

# No local-addressing detuning shift for this run.
local_detuning_offsets = np.zeros(N_sites)

# Effectively global Rabi
def omega_half_t(t):
    ramp_frac = 0.09
    s = np.clip(t / max(t_sweep, np.finfo(float).eps), 0.0, 1.0)

    def smoothstep5(u):
        u = np.clip(u, 0.0, 1.0)
        return 10.0 * u**3 - 15.0 * u**4 + 6.0 * u**5

    if s < ramp_frac:
        env = smoothstep5(s / ramp_frac)
    elif s > 1.0 - ramp_frac:
        env = smoothstep5((1.0 - s) / ramp_frac)
    else:
        env = 1.0

    return 0.5 * Omega * env

# 全局 Rydberg 失谐：平滑往返扫频 -10 MHz -> +10 MHz -> -10 MHz
def delta_t(t):
    s = np.clip(t / max(t_sweep, np.finfo(float).eps), 0.0, 1.0)
    delta_mid = 0.5 * (delta_start + delta_end)
    delta_amp = 0.5 * (delta_end - delta_start)
    return delta_mid - delta_amp * np.cos(2.0 * np.pi * s)

# 局部 addressing shift；这里全部为 0
def delta_local(t, ind):
    return local_detuning_offsets[ind]

# 1. 几何 + 能级结构：3x3 方格，每个格点是 |1>(基态) / |r>(Rydberg) 两能级
geom = make_square_lattice(3, 3, spacing_um=a_um)

# 2. 自定义 sweep：global Delta(t) 和 local address shift 分开输入
protocol = rg.SweepProtocol(
    t_gate=t_sweep,
    omega_half_fn=omega_half_t,
    delta_fn=delta_t,
    address_fn=delta_local,
    n_steps=120,
)
system = rg.RydbergSystem.from_lattice(
    geom, "1r",
    interaction=InteractionSpec(C6=C6_70s, mode="nn"),       # 近邻 VdW；TN 勿用默认 all
    protocol=protocol,
)
params = system.unpack_params([])
t_eval = np.linspace(0.0, t_sweep, 7)

protocol.plot(system=system, params=params, savefig=False)


In [ ]:
# Benchmark 公共量：只放数据容器和几何尺寸，不放算法开关。
dt_tn = 0.2 / Omega  # seconds; 0.2 * Omega^{-1}, same time unit as t_sweep/t_eval
coords = np.asarray(geom.coords)
x_vals = np.unique(coords[:, 0])
y_vals = np.unique(coords[:, 1])
Lx, Ly = len(x_vals), len(y_vals)

# 确认 TN lowering 继承 system 的最近邻 interaction pair list。
tn_spec = tn_lattice_spec_from_system(system)
pair_distances = np.asarray([
    np.linalg.norm(coords[int(i)] - coords[int(j)])
    for i, j, _ in tn_spec.vdw_pairs
])
print(f"TN pairs: {len(tn_spec.vdw_pairs)} bonds; nearest-neighbor only = {np.allclose(pair_distances, a_um)}")

benchmark_results = {}
benchmark_rows = []


## 3. Exact state-vector baseline

下面两个 block 使用 `backend="exact"`。

- exact 后端直接保存每个 `t_eval` 的态矢量，每段对完整$H$做精确矩阵指数，无 Trotter 误差
- 仅有认定 $H(t)$ 在段内常数的离散误差
- 平均 Rydberg occupation 和逐格点 occupation 都用 `system.expectation(...)` 从态矢量读出。这个结果作为后面所有近似后端的数值基准。


In [ ]:
# 3a. 精确后端：states 是态矢量，用 system.expectation 读 Rydberg 均值和逐格点 occupation。
method = "exact"
_t0 = time.perf_counter()
res = rg.simulate(system, [], "all_ground", backend="exact", t_eval=t_eval)
exact_elapsed = time.perf_counter() - _t0

exact_n_mean = np.asarray([
    system.expectation("sum_nr", psi) / system.N
    for psi in res.states
])
exact_n_i = np.asarray([
    [system.expectation(f"n_r_{i}", psi) for i in range(system.N)]
    for psi in res.states
])

benchmark_results[method] = {
    "label": "Exact state-vector",
    "status": "ok",
    "elapsed_s": exact_elapsed,
    "times": np.asarray(res.times),
    "n_mean": exact_n_mean,
    "n_i": exact_n_i,
    "max_abs_diff_n_mean": 0.0,
    "max_abs_diff_n_i": 0.0,
}
benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
benchmark_rows.append({
    "method": method,
    "status": "ok",
    "elapsed_s": exact_elapsed,
    "max_abs_diff_n_mean": 0.0,
    "max_abs_diff_n_i": 0.0,
    "error": "",
})

print(f"Exact elapsed: {exact_elapsed:.3f} s")
print("time(us)  exact <n_r>")
for t, exact_val in zip(res.times * 1e6, exact_n_mean):
    print(f"{t:7.3f}   {exact_val:10.4f}")


单原子完全绝热到末时刻 dressed ground state 的裸 |r> 占据上限也只有
    $$
    \frac{1+\Delta/\sqrt{\Delta^2+\Omega^2}}{2}
    \frac{1+10/\sqrt{10^2+3.8^2}}{2}
    \approx 0.96739
    $$
因为最后 Ω 还开着，Δ_end/Ω≈2.63 不是无限大。

In [ ]:
# 3b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if exact_n_i is None:
    print("exact has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res.times, exact_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("Exact per-site Rydberg occupation", y=1.05)
    plt.show()


## 4. TeNPy CPU MPS-TDVP (`mps`)

下面两个 block 使用 `backend="mps"`。它把 3×3 二维晶格按 snake ordering 拉平成一维 MPS，然后用 TeNPy TDVP 做时间演化。主要参数是 `chi_max` 和 `dt`；这里所有 interaction 来自前面的 `system`，也就是最近邻 `nn` pair list。


In [ ]:
# 4a. TeNPy MPS-TDVP (`backend="mps"`)：TN 后端把 <n_r> 和每个格点 <n_i> 记录在 metadata["obs"] 里。
method = "mps"
try:
    _t0 = time.perf_counter()
    res_tenpy = rg.simulate(
        system, [], "all_ground",
        backend="mps", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"chi_max": 32, "dt": dt_tn, "svd_min": 1e-10},
    )
    tenpy_elapsed = time.perf_counter() - _t0
    tenpy_n_mean = np.asarray(res_tenpy.metadata["obs"]["n_mean"])
    tenpy_n_i = np.asarray(res_tenpy.metadata["obs"]["n_i"])

    tenpy_diff_mean = float(np.max(np.abs(tenpy_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    tenpy_diff_i = float(np.max(np.abs(tenpy_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "TeNPy MPS-TDVP", "status": "ok", "elapsed_s": tenpy_elapsed, "times": np.asarray(res_tenpy.times), "n_mean": tenpy_n_mean, "n_i": tenpy_n_i, "max_abs_diff_n_mean": tenpy_diff_mean, "max_abs_diff_n_i": tenpy_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": tenpy_elapsed, "max_abs_diff_n_mean": tenpy_diff_mean, "max_abs_diff_n_i": tenpy_diff_i, "error": ""})

    print(f"TeNPy elapsed: {tenpy_elapsed:.3f} s")
    print("time(us)  exact <n_r>  TeNPy <n_r>")
    for t, exact_val, tn_val in zip(t_eval * 1e6, exact_n_mean, tenpy_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {tn_val:10.4f}")
    print(f"max |delta <n_r>| = {tenpy_diff_mean:.3e}; max |delta <n_i>| = {tenpy_diff_i:.3e}")
except Exception as exc:
    tenpy_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    tenpy_n_mean = None
    tenpy_n_i = None
    tenpy_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "TeNPy MPS-TDVP", "status": "failed", "elapsed_s": tenpy_elapsed, "error": tenpy_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": tenpy_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": tenpy_error})
    print(f"TeNPy failed after {tenpy_elapsed:.3f} s")
    print(tenpy_error)


In [ ]:
# 4b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if tenpy_n_i is None:
    print("mps has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_tenpy.times, tenpy_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("TeNPy MPS-TDVP per-site Rydberg occupation", y=1.05)
    plt.show()


## 5. cuTensorNet / CuPy GPUTN (`gputn`)

下面两个 block 使用 `backend="gputn"`。它需要 CuPy、cuQuantum 和可见 NVIDIA GPU。这个 block 保持和其它 TN 一样的 `system` 最近邻 interaction；如果本机缺 CUDA 依赖，会把失败原因写进 benchmark 表，不中断后续 block。


In [ ]:
# 5a. cuTensorNet/CuPy GPUTN：真正 CUDA tensor-network kernel，可用时计入同一 benchmark。
method = "gputn"
try:
    _t0 = time.perf_counter()
    res_gputn = rg.simulate(
        system, [], "all_ground",
        backend="gputn", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"chi_max": 32, "dt": dt_tn, "svd_min": 1e-10, "require_gpu": True, "kernel": "auto"},
    )
    gputn_elapsed = time.perf_counter() - _t0
    gputn_n_mean = np.asarray(res_gputn.metadata["obs"]["n_mean"])
    gputn_n_i = np.asarray(res_gputn.metadata["obs"]["n_i"])

    gputn_diff_mean = float(np.max(np.abs(gputn_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    gputn_diff_i = float(np.max(np.abs(gputn_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "cuTensorNet GPUTN", "status": "ok", "elapsed_s": gputn_elapsed, "times": np.asarray(res_gputn.times), "n_mean": gputn_n_mean, "n_i": gputn_n_i, "max_abs_diff_n_mean": gputn_diff_mean, "max_abs_diff_n_i": gputn_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": gputn_elapsed, "max_abs_diff_n_mean": gputn_diff_mean, "max_abs_diff_n_i": gputn_diff_i, "error": ""})

    print(f"GPUTN elapsed: {gputn_elapsed:.3f} s")
    print("time(us)  exact <n_r>  GPUTN <n_r>")
    for t, exact_val, gpu_val in zip(t_eval * 1e6, exact_n_mean, gputn_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {gpu_val:10.4f}")
    print(f"max |delta <n_r>| = {gputn_diff_mean:.3e}; max |delta <n_i>| = {gputn_diff_i:.3e}")
except Exception as exc:
    gputn_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    gputn_n_mean = None
    gputn_n_i = None
    gputn_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "cuTensorNet GPUTN", "status": "failed", "elapsed_s": gputn_elapsed, "error": gputn_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": gputn_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": gputn_error})
    print(f"GPUTN failed after {gputn_elapsed:.3f} s")
    print(gputn_error)


In [ ]:
# 5b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if gputn_n_i is None:
    print("gputn has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_gputn.times, gputn_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("cuTensorNet GPUTN per-site Rydberg occupation", y=1.05)
    plt.show()


## 6. YASTN finite-PEPS (`peps`)

下面两个 block 使用 `backend="peps"`。这是 YASTN finite-PEPS 路径，直接保留二维方格几何；默认用 NTU 更新和 BP measurement。设置 `use_cuda=True, yastn_backend="torch", device="cuda"` 后可走 YASTN 的 GPU backend。


In [ ]:
# 6a. YASTN finite-PEPS：二维 PEPS 主路径。
method = "peps"
try:
    _t0 = time.perf_counter()
    res_peps = rg.simulate(
        system, [], "all_ground",
        backend="peps", t_eval=t_eval, observables=["n_mean", "n_i"],
        backend_options={"chi_max": 8, "dt": dt_tn, "svd_min": 1e-10, "measurement_environment": "bp", "use_cuda": False},
    )
    peps_elapsed = time.perf_counter() - _t0
    peps_n_mean = np.asarray(res_peps.metadata["obs"]["n_mean"])
    peps_n_i = np.asarray(res_peps.metadata["obs"]["n_i"])

    peps_diff_mean = float(np.max(np.abs(peps_n_mean - exact_n_mean))) if "exact_n_mean" in globals() else np.nan
    peps_diff_i = float(np.max(np.abs(peps_n_i - exact_n_i))) if "exact_n_i" in globals() else np.nan
    benchmark_results[method] = {"label": "YASTN finite-PEPS", "status": "ok", "elapsed_s": peps_elapsed, "times": np.asarray(res_peps.times), "n_mean": peps_n_mean, "n_i": peps_n_i, "max_abs_diff_n_mean": peps_diff_mean, "max_abs_diff_n_i": peps_diff_i}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "ok", "elapsed_s": peps_elapsed, "max_abs_diff_n_mean": peps_diff_mean, "max_abs_diff_n_i": peps_diff_i, "error": ""})

    print(f"PEPS elapsed: {peps_elapsed:.3f} s")
    print("time(us)  exact <n_r>  PEPS <n_r>")
    for t, exact_val, p_val in zip(t_eval * 1e6, exact_n_mean, peps_n_mean):
        print(f"{t:7.3f}   {exact_val:10.4f}  {p_val:10.4f}")
    print(f"max |delta <n_r>| = {peps_diff_mean:.3e}; max |delta <n_i>| = {peps_diff_i:.3e}")
except Exception as exc:
    peps_elapsed = time.perf_counter() - _t0 if "_t0" in globals() else np.nan
    peps_n_mean = None
    peps_n_i = None
    peps_error = f"{type(exc).__name__}: {exc}"
    benchmark_results[method] = {"label": "YASTN finite-PEPS", "status": "failed", "elapsed_s": peps_elapsed, "error": peps_error}
    benchmark_rows = [row for row in benchmark_rows if row["method"] != method]
    benchmark_rows.append({"method": method, "status": "failed", "elapsed_s": peps_elapsed, "max_abs_diff_n_mean": np.nan, "max_abs_diff_n_i": np.nan, "error": peps_error})
    print(f"PEPS failed after {peps_elapsed:.3f} s")
    print(peps_error)


In [ ]:
# 6b. 每个时刻的 3x3 Rydberg occupation color map：使用当前算法的逐格点 <n_r_i>。
if peps_n_i is None:
    print("peps has no successful data to plot.")
else:
    fig_h, axes_h = plt.subplots(1, len(t_eval), figsize=(2.2 * len(t_eval), 2.5), constrained_layout=True)
    axes_h = np.atleast_1d(axes_h)

    for ax_h, t, occ in zip(axes_h, res_peps.times, peps_n_i):
        grid = occ.reshape(Lx, Ly)
        im = ax_h.imshow(grid.T, origin="lower", vmin=0.0, vmax=1.0, cmap="viridis")
        ax_h.set_title(f"{t * 1e6:.2f} us")
        ax_h.set_xticks(range(Lx))
        ax_h.set_yticks(range(Ly))
        ax_h.set_xlabel("x")
        if ax_h is axes_h[0]:
            ax_h.set_ylabel("y")
        else:
            ax_h.set_yticklabels([])
        for ix in range(Lx):
            for iy in range(Ly):
                val = grid[ix, iy]
                txt_color = "white" if val > 0.55 else "black"
                ax_h.text(ix, iy, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=8)

    fig_h.colorbar(im, ax=axes_h, shrink=0.82, label=r"$\langle n_r \rangle$")
    fig_h.suptitle("YASTN finite-PEPS per-site Rydberg occupation", y=1.05)
    plt.show()


## 7. 汇总对比图

这个 block 汇总已经运行过的算法，列出状态、墙钟时间、相对 exact 的最大平均 occupation 误差和最大逐格点 occupation 误差，并画运行时间、平均 Rydberg occupation 曲线和误差柱状图。失败后端会保留在文字表格中。


In [ ]:
print("full benchmark table:")
print(f"{'method':<14} {'status':<8} {'time(s)':>10} {'max|dn_mean|':>14} {'max|dn_i|':>12}")
for row in benchmark_rows:
    elapsed = row.get("elapsed_s", np.nan)
    err_mean = row.get("max_abs_diff_n_mean", np.nan)
    err_i = row.get("max_abs_diff_n_i", np.nan)
    elapsed_s = f"{elapsed:.3f}" if np.isfinite(elapsed) else "nan"
    err_mean_s = f"{err_mean:.3e}" if np.isfinite(err_mean) else "nan"
    err_i_s = f"{err_i:.3e}" if np.isfinite(err_i) else "nan"
    print(f"{row['method']:<14} {row['status']:<8} {elapsed_s:>10} {err_mean_s:>14} {err_i_s:>12}")
    if row.get("error"):
        print("  error:", row["error"])

ok_methods = [method for method, data in benchmark_results.items() if data.get("status") == "ok"]
if not ok_methods:
    print("No successful backend results to summarize.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

    elapsed = np.asarray([benchmark_results[m]["elapsed_s"] for m in ok_methods], dtype=float)
    axes[0].bar(ok_methods, elapsed, color="tab:blue", alpha=0.8)
    axes[0].set_ylabel("wall time (s)")
    axes[0].set_title("Runtime")
    axes[0].tick_params(axis="x", rotation=45)

    for method in ok_methods:
        data = benchmark_results[method]
        axes[1].plot(
            np.asarray(data["times"]) * 1e6,
            np.asarray(data["n_mean"]),
            marker="o",
            linewidth=1.6,
            label=method,
        )
    axes[1].set_xlabel("time (us)")
    axes[1].set_ylabel(r"mean $\langle n_r \rangle$")
    axes[1].set_title("Mean Rydberg occupation")
    axes[1].legend(fontsize=8)

    err_mean = np.asarray([benchmark_results[m].get("max_abs_diff_n_mean", np.nan) for m in ok_methods], dtype=float)
    axes[2].bar(ok_methods, err_mean, color="tab:orange", alpha=0.85)
    axes[2].set_yscale("symlog", linthresh=1e-12)
    axes[2].set_ylabel("max abs diff vs exact")
    axes[2].set_title("Mean occupation error")
    axes[2].tick_params(axis="x", rotation=45)
    plt.show()

    fastest = min(ok_methods, key=lambda m: benchmark_results[m]["elapsed_s"])
    comparable = [m for m in ok_methods if m != "exact" and np.isfinite(benchmark_results[m].get("max_abs_diff_n_mean", np.nan))]
    print(f"fastest successful backend: {fastest} ({benchmark_results[fastest]['elapsed_s']:.3f} s)")
    if comparable:
        best = min(comparable, key=lambda m: benchmark_results[m]["max_abs_diff_n_mean"])
        worst = max(comparable, key=lambda m: benchmark_results[m]["max_abs_diff_n_mean"])
        print(f"most consistent mean occupation: {best} ({benchmark_results[best]['max_abs_diff_n_mean']:.3e})")
        print(f"largest mean-occupation deviation: {worst} ({benchmark_results[worst]['max_abs_diff_n_mean']:.3e})")
